# Module 5 — Beyond Pairwise: **Hypergraph Networks** (and the loop closes)
### TIFR ML School 2026 · *Sets, Graphs & Symmetry for High-Energy Physics* · **Capstone**

Every model so far connects particles in **pairs**: a graph edge (Module 2), a soft attention edge (Module 3),
a symmetry-constrained edge (Module 4). But some physics is irreducibly **higher-order** — it lives in *groups*,
not pairs:
- a boosted top decays to **three** prongs; a $W$ to **two** — group membership, not pairwise distance;
- an **event** is a *set of jets*, each a *set of particles* — a hierarchy of sets (a "set of sets");
- **assignment**: which particles came from which parent? — grouping is the whole task;
- **higher-order energy correlators** ($N$-point) measure genuine $N$-particle structure.

A **hyperedge** binds a whole *group* of nodes at once. The network that message-passes over hyperedges is a
**hypergraph neural network**, and the clean modern formulation — **AllSet** — turns out to be **two nested
DeepSets**. The atom we started the course with in Module 1 reappears at the very top. The loop closes.

**What you will do, end-to-end:**
1. Define **hypergraphs** and the **incidence** structure; see why pairs are not enough.
2. Implement an **AllSet layer** by **reusing the Module-1 DeepSets aggregator twice** (nodes→hyperedges→nodes).
3. Build jets as hypergraphs and **train** a hypergraph tagger for top-vs-QCD.
4. **Close the loop:** collapse to a single hyperedge and recover DeepSets (Module 1) *exactly*.
5. **Capstone:** the one table that unifies all five modules; and an outlook to foundation models.

**Dependencies:** `numpy torch torch_geometric matplotlib scikit-learn uproot awkward`.

## 1 · Hypergraphs and incidence

A graph edge joins exactly two nodes. A **hyperedge** joins **any number** of nodes — a *group*. A hypergraph is
$(\mathcal V,\mathcal E)$ with each $e\in\mathcal E$ a subset of nodes. Its structure is the **incidence**
$H\in\{0,1\}^{|\mathcal V|\times|\mathcal E|}$, $H_{ve}=1$ iff node $v$ belongs to hyperedge $e$. We store it
sparsely as two parallel index lists `hyperedge_index = [node_idx ; edge_idx]` (one column per membership).

Why this is the right object for the bullets above: a hyperedge can *be* a subjet (the group of particles in one
prong), a jet (the group of particles in it), or a candidate parent (the group of decay products) — relationships
that a pairwise edge can only approximate.

## 2 · The payoff: an AllSet layer is **two nested DeepSets**

How do you message-pass on a hypergraph? **AllSet** (Chien et al., 2022) does it in two permutation-invariant
half-steps — and *each half-step is a Deep Set* (Module 1):

$$
\textbf{V}\!\to\!\textbf{E:}\quad z_e=\rho_E\!\Big(\!\!\sum_{v\in e}\phi_E(h_v)\Big)
\qquad\qquad
\textbf{E}\!\to\!\textbf{V:}\quad h_v'=h_v+\rho_V\!\Big(\!\!\sum_{e\ni v}\phi_V(z_e)\Big)
$$

- **V→E**: each hyperedge summarizes its **member nodes** — a Deep Set over the group.
- **E→V**: each node summarizes the **hyperedges it belongs to** — a Deep Set over its memberships.

That's the whole idea, and it is exactly $\rho(\sum\phi(\cdot))$ twice — the Module-1 aggregator, reused. Two
special cases make the unification vivid:
- **one hyperedge containing every node** ⟶ V→E is a single global pool ⟶ the layer **is DeepSets / a PFN** (§4).
- **every hyperedge a pair** $\{i,j\}$ ⟶ you recover ordinary **graph** message passing (Module 2).

So DeepSets and GNNs are both *special cases* of a hypergraph network. We close the loop by building it from the
Module-1 atom.

In [ ]:
%matplotlib inline
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch_geometric.utils import scatter
from torch_geometric.nn import global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

torch.manual_seed(0); np.random.seed(0)
device = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print("device", device)

def mlp(i, o, h=64):
    return nn.Sequential(nn.Linear(i, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, o))

def knn_graph_native(x, k, batch):
    """Within-graph k-NN (pure torch) -> edge_index [source(neighbour), target(centre)]. Used to BUILD hyperedges."""
    with torch.no_grad():
        N = x.size(0)
        x2 = (x * x).sum(1, keepdim=True)
        d2 = (x2 + x2.t() - 2 * x @ x.t()).masked_fill(~(batch[None] == batch[:, None]), float("inf"))
        d2.fill_diagonal_(float("inf"))
        kk = min(k, max(N - 1, 1))
        knn_d, idx = d2.topk(kk, dim=1, largest=False)
        valid = torch.isfinite(knn_d)
        centre = torch.arange(N, device=x.device).unsqueeze(1).expand(-1, kk)
        return torch.stack([idx[valid], centre[valid]], dim=0)

In [ ]:
class DeepSetsAgg(nn.Module):
    """The Module-1 atom:  rho( sum_over_group phi(x) ).  Used twice to make one AllSet layer."""
    def __init__(self, in_dim, out_dim, reduce="mean"):
        super().__init__()
        self.phi = mlp(in_dim, out_dim)
        self.rho = mlp(out_dim, out_dim)
        self.reduce = reduce
    def forward(self, x, src, index, dim_size):
        # gather phi(x) for each membership (src), pool by `index` (the group id), decode with rho
        agg = scatter(self.phi(x)[src], index, dim=0, dim_size=dim_size, reduce=self.reduce)
        return self.rho(agg)

class AllSetLayer(nn.Module):
    """One hypergraph message-passing layer = TWO nested Deep Sets (V->E then E->V)."""
    def __init__(self, dim):
        super().__init__()
        self.v2e = DeepSetsAgg(dim, dim)        # nodes  -> hyperedges
        self.e2v = DeepSetsAgg(dim, dim)        # hyperedges -> nodes
        self.norm = nn.LayerNorm(dim)
    def forward(self, h, node_idx, edge_idx, num_nodes, num_edges):
        z = self.v2e(h, node_idx, edge_idx, num_edges)        # each hyperedge aggregates its member nodes
        h = self.norm(h + self.e2v(z, edge_idx, node_idx, num_nodes))   # each node aggregates its hyperedges
        return h, z

# shape smoke + a symmetry check on a hand-made hypergraph: two hyperedges e0={0,1}, e1={2,3}
layer = AllSetLayer(8)
h = torch.randn(2, 8).repeat_interleave(2, 0)            # give each group identical members: [a,a,b,b]
node_idx = torch.tensor([0, 1, 2, 3]); edge_idx = torch.tensor([0, 0, 1, 1])
h2, z = layer(h, node_idx, edge_idx, num_nodes=4, num_edges=2)
print("h:", tuple(h2.shape), " z(hyperedges):", tuple(z.shape))
# the aggregation is permutation-invariant within a group, so identical members stay identical through V->E->V
same = torch.allclose(h2[0], h2[1], atol=1e-6) and torch.allclose(h2[2], h2[3], atol=1e-6)
print("group members treated symmetrically (identical in -> identical out):", same)

### 2.1 · Grounding it in code — the incidence matrix, and why a *pair* is not enough

Before touching jets, make §1–§2 concrete on a tiny hand-built hypergraph. Six nodes, three hyperedges

$$e_0=\{0,1,2\},\qquad e_1=\{2,3\},\qquad e_2=\{3,4,5\},$$

with $e_0,e_1$ sharing node 2 and $e_1,e_2$ sharing node 3. We build the **incidence matrix**
$H\in\{0,1\}^{6\times 3}$ ($H_{ve}=1$ iff node $v\in e$), read off the two quantities a hypergraph layer sums
over — **node degree** $d_v=\sum_e H_{ve}$ (how many groups a node is in) and **hyperedge size**
$|e|=\sum_v H_{ve}$ — and then form the **clique expansion** $A=\mathbf 1[HH^{\top}>0]$: the ordinary graph a
pairwise model would see. The punchline is the last line. The clique graph turns the size-3 group $e_0$ into a
*triangle*, and **a triangle of pairwise edges is indistinguishable from three unrelated pairs**. The "these three
belong together" information is exactly what a hyperedge keeps and a graph throws away.

In [ ]:
# A tiny hypergraph by hand -> incidence matrix, degrees, sizes, and the (lossy) clique expansion.
hyperedges = [[0, 1, 2], [2, 3], [3, 4, 5]]                 # e0 (triple), e1 (pair), e2 (triple)
Nn, Ne = 6, len(hyperedges)
H = np.zeros((Nn, Ne), dtype=int)
for e, members in enumerate(hyperedges):
    H[members, e] = 1                                       # H_{ve} = 1  iff node v is in hyperedge e
print("incidence H (6 nodes x 3 hyperedges):\n", H)
print("node degrees  d_v =", H.sum(1), "  (memberships per node)")
print("hyperedge sizes |e| =", H.sum(0), "  (nodes per group)")

A = (H @ H.T); np.fill_diagonal(A, 0); A = (A > 0).astype(int)     # share a hyperedge -> a pairwise edge
print("\nclique-expansion adjacency A (what a pairwise GNN sees):\n", A)
print(f"clique graph has {A.sum() // 2} edges; the size-3 group e0 became a triangle. A triangle of pairs is")
print("indistinguishable from one 3-node group -- that lost 'togetherness' is why hyperedges exist.")

# draw it: incidence heatmap (left) + the hypergraph with each group as a coloured star (right)
posN = np.array([[0, 2], [1, 2.4], [0.5, 1.2], [2, 1.1], [3, 1.7], [2.7, 0.4]], float)   # node coords for the picture
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].imshow(H, cmap="Blues", aspect="auto", vmin=0, vmax=1)
ax[0].set_xticks(range(Ne)); ax[0].set_xticklabels([f"$e_{e}$" for e in range(Ne)])
ax[0].set_yticks(range(Nn)); ax[0].set_yticklabels([f"node {v}" for v in range(Nn)])
ax[0].set_title("incidence matrix $H$")
for v in range(Nn):
    for e in range(Ne):
        ax[0].text(e, v, H[v, e], ha="center", va="center", color="crimson", fontsize=9)
cols = ["tab:blue", "tab:orange", "tab:green"]
for e, members in enumerate(hyperedges):
    c = posN[members].mean(0)                               # hyperedge centroid, star out to its members
    for v in members:
        ax[1].plot([c[0], posN[v, 0]], [c[1], posN[v, 1]], color=cols[e], lw=7, alpha=0.25, solid_capstyle="round")
    ax[1].scatter(*c, marker="s", s=60, color=cols[e], label=f"$e_{e}$  (|e|={len(members)})", zorder=3)
ax[1].scatter(posN[:, 0], posN[:, 1], s=280, color="white", edgecolor="black", zorder=4)
for v in range(Nn):
    ax[1].text(posN[v, 0], posN[v, 1], str(v), ha="center", va="center", zorder=5)
ax[1].set_title("the hypergraph (each colour = one group)"); ax[1].legend(loc="lower left", fontsize=8); ax[1].axis("off")
plt.tight_layout(); plt.show()

### 2.2 · Verify each half-step is permutation-invariant

Module 1's whole point was that $\rho\!\big(\sum_i \phi(x_i)\big)$ is **permutation-invariant**: the output does
not depend on the order of the inputs. AllSet inherits that guarantee *twice*. We check it numerically — in the
same spirit as the invariance test in Module 1 and the equivariance tests in Module 4 — by running the **V→E**
step, then randomly **shuffling the order of the memberships** (which reorders the members inside every
hyperedge), and confirming the hyperedge embeddings $z_e$ come back **bit-for-bit identical**. Same for the full
layer. This is not a lucky initialization; it is structural, because `scatter(..., reduce="mean")` is a symmetric
function of its inputs.

In [ ]:
# Permutation invariance: reordering members within hyperedges must not change the hyperedge embeddings.
torch.manual_seed(1)
demo = AllSetLayer(8)
node_idx = torch.tensor([0, 1, 2,  2, 3,  3, 4, 5])         # memberships of e0, e1, e2 as columns
edge_idx = torch.tensor([0, 0, 0,  1, 1,  2, 2, 2])
h_demo = torch.randn(Nn, 8)

z = demo.v2e(h_demo, node_idx, edge_idx, Ne)                # V->E: one embedding per hyperedge
perm = torch.randperm(node_idx.numel())                     # reorder the membership columns == reorder members
z_shuf = demo.v2e(h_demo, node_idx[perm], edge_idx[perm], Ne)
print("V->E invariant to member order  :", torch.allclose(z, z_shuf, atol=1e-6),
      f"   (max |dz| = {(z - z_shuf).abs().max():.1e})")

h1, _ = demo(h_demo, node_idx, edge_idx, Nn, Ne)
h2, _ = demo(h_demo, node_idx[perm], edge_idx[perm], Nn, Ne)
print("full AllSet layer invariant     :", torch.allclose(h1, h2, atol=1e-6),
      "  <- the Module-1 DeepSets guarantee, now holding twice (V->E and E->V)")

### 2.3 · The mathematics of hypergraph message passing

§2 gave the AllSet layer as "two nested Deep Sets." Let us place it in the full landscape and write the maths
carefully. A hypergraph is $\mathcal G=(\mathcal V,\mathcal E)$ with $n=|\mathcal V|$ nodes, $m=|\mathcal E|$
hyperedges, and incidence $H\in\{0,1\}^{n\times m}$, $H_{ve}=1\Leftrightarrow v\in e$. Two neighbourhoods matter:
the **members** of a hyperedge $V_e=\{v:H_{ve}=1\}$ and the **incident hyperedges** of a node
$E_v=\{e:H_{ve}=1\}$.

**The general layer = two permutation-invariant multiset maps.** *Every* hypergraph message-passing layer can be
written as

$$
\underbrace{m_e=f_{\mathcal V\to\mathcal E}\big(\{\!\{\,h_v:v\in e\,\}\!\}\big)}_{\text{V}\to\text{E: gather a hyperedge's members}}
\qquad
\underbrace{h_v'=f_{\mathcal E\to\mathcal V}\big(h_v,\ \{\!\{\,m_e:e\ni v\,\}\!\}\big)}_{\text{E}\to\text{V: gather a node's hyperedges}}
$$

where $\{\!\{\cdot\}\!\}$ is a **multiset** and each $f$ must be invariant to the order of its arguments — because a
hyperedge's members and a node's incident hyperedges are *unordered* sets (we checked this numerically in §2.2). By
the Deep Sets theorem (Module 1), the most general such map is $\rho\!\big(\sum\phi(\cdot)\big)$ — which is exactly
AllSet.

**The matrix skeleton.** Stack node features as $X\in\mathbb R^{n\times d}$ and hyperedge messages as
$Z\in\mathbb R^{m\times d}$. Whatever the nonlinearities, the *aggregation* in each stage is a multiply by the
incidence matrix:

$$
Z \;=\; \underbrace{H^{\top}}_{\text{V}\to\text{E}}\,\phi_{\mathcal E}(X),
\qquad\qquad
X' \;=\; \underbrace{H}_{\text{E}\to\text{V}}\,\phi_{\mathcal V}(Z),
$$

up to per-edge and per-node normalization. **V→E is a multiply by $H^{\top}$; E→V is a multiply by $H$.** Every
paradigm below is just a different choice of the maps $\phi,\rho$ and of that normalization.

**Symmetry.** Relabelling nodes by a permutation matrix $P$ and hyperedges by $Q$ sends $H\mapsto PHQ^{\top}$ and
$X\mapsto PX$. The layer is **equivariant** to node relabelling and **invariant** to hyperedge relabelling:
$X'\mapsto PX'$ and $Z\mapsto QZ$. This is the same permutation symmetry that has run through the whole course, now
expressed on the incidence matrix.

### 2.4 · Four paradigms — and how they collapse to one skeleton

| Paradigm | what the two stages are | one-line maths |
|---|---|---|
| **Expansion → GNN** | project to a graph, run Module 2 | clique adjacency $A=HWH^{\top}-D_v$ |
| **Spectral — HGNN** | normalized **mean** + fixed linear | $X'=\sigma(\Theta\,X\,\Theta_w)$ |
| **Spatial — AllSet** | learnable **Deep Sets**, twice | $\rho\!\big(\sum\phi(\cdot)\big)$ per stage |
| **Attention — AllSet-Transformer** | learnable **attention** pool (PMA) | $m_e=\sum_{v\in e}\alpha_{ve}\,Vh_v$ |

**1 · Expansions (Zhou et al., 2006).** Replace each hyperedge by a clique and run an ordinary GNN. With hyperedge
weights $W=\mathrm{diag}(w_e)$, the co-membership adjacency is $A=HWH^{\top}-D_v$ — its off-diagonal $(i,j)$ counts
the hyperedges shared by $i$ and $j$. *Lossy:* a size-3 group and a triangle of pairs give the same $A$ (§2.1). The
**star** expansion instead keeps the bipartite node–hyperedge graph whose biadjacency **is** $H$; message passing
on it is precisely the two-stage scheme — and loses nothing.

**2 · Spectral — HGNN (Feng et al., 2019).** With node degrees $D_v=\mathrm{diag}(HW\mathbf 1)$ and edge degrees
$D_e=\mathrm{diag}(H^{\top}\mathbf 1)$, Zhou's normalized hypergraph Laplacian is $\Delta=I-\Theta$ with

$$
\Theta \;=\; D_v^{-1/2}\,H\,W\,D_e^{-1}\,H^{\top}\,D_v^{-1/2},
\qquad
\text{HGNN layer:}\quad X^{(l+1)}=\sigma\!\big(\Theta\,X^{(l)}\,\Theta_w\big).
$$

Read $\Theta$ right-to-left: $H^{\top}$ (V→E) → $D_e^{-1}$ (average the members) → $H$ (E→V) → symmetric degree
normalization. **HGNN is the two-stage scheme with a fixed mean pool and a single linear $\phi=\Theta_w$.**

**3 · Spatial two-stage (HNHN 2020; UniGNN 2021; AllSet, Chien et al. 2022).** Keep the two stages but make both
$f$ *learnable* multiset functions. UniGNN: $h_e=\tfrac1{|e|}\sum_{v\in e}h_v$ then
$h_v'=W\,\mathrm{AGG}_{e\ni v}\,h_e$. **AllSet:** both stages are Deep Sets / Set-Transformers — the most expressive
permutation-invariant choice, i.e. §2's $\rho(\sum\phi(\cdot))$ applied twice.

**4 · Attention — AllSet-Transformer (Chien et al., 2022; Set Transformer, Lee et al. 2019).** Replace the pool by
learned attention with a trainable seed query $s$ (Set Transformer's PMA):

$$
\alpha_{ve}=\frac{\exp\langle s,\,Wh_v\rangle}{\sum_{u\in e}\exp\langle s,\,Wh_u\rangle},
\qquad
m_e=\sum_{v\in e}\alpha_{ve}\,V h_v .
$$

That is the Module-3 attention pool, now operating *inside a hyperedge* (Exercise 6 trains it).

**The collapse.** All four share the $H^{\top}$-then-$H$ skeleton, so the models of the whole course are corners of it:
- every hyperedge a **pair** $\{i,j\}$ ⟹ $H$ is an ordinary graph incidence ⟹ **MPNN** (Module 2);
- one hyperedge = **all nodes** ⟹ $H=\mathbf 1$, so V→E is a single global pool ⟹ **DeepSets / PFN** (Module 1);
- **fixed mean + degree norm ⟹ HGNN**;  **learnable Deep Sets ⟹ AllSet**;  **learnable attention ⟹ AllSet-Transformer**.

The cell below verifies the one non-obvious identity — that HGNN's spectral $\Theta$ *is* the explicit two-stage
message pass — on §2.1's toy hypergraph.

In [ ]:
# The paradigms, checked on §2.1's toy hypergraph. Aggregation is always a multiply by the incidence H.
Hf = H.astype(float)                                     # incidence (6 nodes x 3 hyperedges), reused from §2.1
We = np.eye(Hf.shape[1])                                 # hyperedge weights W (identity here)
Dv = np.diag(Hf @ We @ np.ones(Hf.shape[1]))             # node degrees  D_v = diag(H W 1)
De = np.diag(Hf.sum(0))                                  # edge degrees  D_e = diag(|e|)
Dvh = np.diag(1.0 / np.sqrt(np.diag(Dv)))                # D_v^{-1/2}
Dei = np.diag(1.0 / np.diag(De))                         # D_e^{-1}

# HGNN propagation (Zhou / Feng):  Theta = D_v^{-1/2} H W D_e^{-1} H^T D_v^{-1/2}
Theta = Dvh @ Hf @ We @ Dei @ Hf.T @ Dvh
Lap = np.eye(Hf.shape[0]) - Theta                        # normalized hypergraph Laplacian  Delta = I - Theta

# ...the SAME operator, written as an explicit two-stage message pass on a signal `sig`:
rng = np.random.default_rng(0); sig = rng.normal(size=(Hf.shape[0], 4))
sig_n   = Dvh @ sig                                      # symmetric normalization
msg_e   = Dei @ (Hf.T @ sig_n)                           # V->E : multiply by H^T, then mean over each hyperedge's members
sig_out = Dvh @ (Hf @ (We @ msg_e))                      # E->V : multiply by H (scatter back, weighted), then normalize
print("HGNN conv  ==  explicit two-stage (H^T then H) pass :", np.allclose(Theta @ sig, sig_out))

# clique expansion (the graph a pairwise GNN would use):  A_co = H W H^T - D_v
A_co = Hf @ We @ Hf.T - Dv
print("clique adjacency  H W H^T - D_v  ==  §2.1's A       :", np.array_equal((A_co > 1e-9).astype(int), A))
ev = np.linalg.eigvalsh(Lap)
print("normalized Laplacian spectrum lies in [0, 2]        :", bool(np.all(ev > -1e-9) and np.all(ev < 2 + 1e-9)),
      " eigs =", np.round(ev, 3))
print("\nV->E is a multiply by H^T, E->V a multiply by H. HGNN fixes the pool to a normalized mean;")
print("AllSet (§2) makes both stages learnable Deep Sets; the AllSet-Transformer makes the pool learned attention.")

## 3 · Jets as hypergraphs

We turn each jet into a hypergraph: every particle is a node, and for every particle we form a **hyperedge =
that particle together with its $k$ nearest neighbours in $(\eta,\phi)$** (the standard HGNN construction). These
overlapping groups capture *local clusters of particles acting together* — a softer, higher-order version of the
k-NN graph from Module 2. We batch many hypergraphs with a small `Data` subclass that tells PyG how to offset the
incidence indices.

In [ ]:
import os, uproot, awkward as ak

CANDIDATE_PATHS = [
    "../data/JetClass_example_100k.root",
    "/Users/sanmay/Documents/ML_SCHOOLS/MLSCHOOL_2023_ICTS/Main_School/JetDataset/JetClass_example_100k.root",
    "./JetClass_example_100k.root",
]
root_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if root_path is None:
    raise FileNotFoundError("Place JetClass_example_100k.root in ../data/.")

NPART, N_PER_CLS, K_HYPER = 64, 8000, 8
tree = uproot.open(root_path)["tree"]
br = tree.arrays(["part_px","part_py","part_pz","part_energy","part_deta","part_dphi",
                  "label_QCD","label_Tbqq","label_Tbl"])
is_qcd = ak.to_numpy(br["label_QCD"]).astype(bool)
is_top = ak.to_numpy(br["label_Tbqq"]).astype(bool) | ak.to_numpy(br["label_Tbl"]).astype(bool)
sel = np.concatenate([np.where(is_qcd)[0][:N_PER_CLS], np.where(is_top)[0][:N_PER_CLS]])
labels = np.concatenate([np.zeros(N_PER_CLS), np.ones(N_PER_CLS)]).astype(np.int64)

px,py,pz,e = br["part_px"][sel], br["part_py"][sel], br["part_pz"][sel], br["part_energy"][sel]
deta,dphi  = br["part_deta"][sel], br["part_dphi"][sel]
pt = np.sqrt(px**2 + py**2); dR = np.sqrt(deta**2 + dphi**2)
sumpt, sume = ak.sum(pt, axis=1), ak.sum(e, axis=1)
order = ak.argsort(pt, axis=1, ascending=False)                 # leading particles by pT
def topN(a): return ak.to_numpy(ak.fill_none(ak.pad_none(a[order], NPART, clip=True), 0.0))
feat = [deta, dphi, np.log(pt+1e-8), np.log(e+1e-8), np.log(pt/sumpt+1e-8), np.log(e/sume+1e-8), dR]
X   = np.stack([topN(f) for f in feat], axis=-1).astype(np.float32)
POS = np.stack([topN(deta), topN(dphi)], axis=-1).astype(np.float32)
counts = np.minimum(ak.to_numpy(ak.num(pt, axis=1)), NPART)
realmask = (np.arange(NPART)[None,:] < counts[:,None])
mean, std = X[realmask].mean(0), X[realmask].std(0) + 1e-6
X = (X - mean) / std
print("featurized", len(labels), "jets, up to", NPART, "particles")

In [ ]:
class HyperData(Data):
    """A hypergraph sample. hyperedge_index = [node_idx ; edge_idx]; __inc__ offsets BOTH rows on batching."""
    def __inc__(self, key, value, *args, **kwargs):
        if key == "hyperedge_index":
            return torch.tensor([[self.num_nodes], [self.num_hyperedges]])
        return super().__inc__(key, value, *args, **kwargs)

def build_hypergraph(x, pos, label, k):
    n = x.shape[0]
    ei = knn_graph_native(torch.from_numpy(pos), k, torch.zeros(n, dtype=torch.long))   # [neighbour, centre]
    # hyperedge c = {c} U kNN(c): self-memberships plus neighbour memberships
    node_idx = torch.cat([torch.arange(n), ei[0]])
    edge_idx = torch.cat([torch.arange(n), ei[1]])
    d = HyperData(x=torch.from_numpy(x), hyperedge_index=torch.stack([node_idx, edge_idx]),
                  y=torch.tensor([label], dtype=torch.long))
    d.num_hyperedges = n
    return d

data_list = [build_hypergraph(X[i, :int(counts[i])], POS[i, :int(counts[i])], int(labels[i]), K_HYPER)
             for i in range(len(labels)) if counts[i] >= 2]

import random
random.seed(0); random.shuffle(data_list)
n = len(data_list); a, b = int(0.70*n), int(0.85*n)
loaders = {"train": DataLoader(data_list[:a], batch_size=64, shuffle=True),
           "val":   DataLoader(data_list[a:b], batch_size=128),
           "test":  DataLoader(data_list[b:], batch_size=128)}
bb = next(iter(loaders["train"]))
print("batched:", bb)
print("a batch has", bb.num_graphs, "jets,", bb.x.shape[0], "nodes,",
      int(bb.hyperedge_index[1].max())+1, "hyperedges")

### 3.1 · See a jet as a hypergraph

Abstractions become intuition when you *look*. Below we take one **top** jet and one **QCD** jet from the data we
just featurized, rebuild their $k$-NN hyperedges in the $(\eta,\phi)$ plane, and draw the groups: each faint
coloured star is one hyperedge (a particle plus its $k$ nearest neighbours). Top jets show a few dense clusters —
the decay prongs — that hyperedges naturally wrap; QCD jets are more diffuse. The third panel is the **incidence
matrix** of the top jet (particles × hyperedges — the real version of §2.1's toy $H$), and the last shows how many
hyperedges each particle ends up in (its **degree** $d_v$): a few "hub" particles sit inside many groups.

In [ ]:
# Visualize real jets as hypergraphs (positions are raw Delta-eta, Delta-phi; k-NN hyperedges rebuilt for the picture).
def jet_hyperedges(pos, k):
    n = pos.shape[0]
    ei = knn_graph_native(torch.from_numpy(pos), k, torch.zeros(n, dtype=torch.long))   # [neighbour, centre]
    node_idx = torch.cat([torch.arange(n), ei[0]]); edge_idx = torch.cat([torch.arange(n), ei[1]])
    Hd = np.zeros((n, n), dtype=int); Hd[node_idx, edge_idx] = 1                          # incidence (n x n)
    return node_idx, edge_idx, Hd

top_i = next(i for i in range(len(labels)) if labels[i] == 1 and counts[i] >= 25)         # a busy top jet
qcd_i = next(i for i in range(len(labels)) if labels[i] == 0 and counts[i] >= 15)         # a QCD jet
fig, ax = plt.subplots(1, 4, figsize=(15, 3.7))
for col, (idx, name) in enumerate([(top_i, "top jet"), (qcd_i, "QCD jet")]):
    n = int(counts[idx]); pos = POS[idx, :n]; logpt = X[idx, :n, 2]                        # feature 2 ~ log pT (scaled)
    ni, ei, _ = jet_hyperedges(pos, K_HYPER)
    for c in range(0, n, max(1, n // 6)):                                                  # a few hyperedges, coloured by centre
        for v in ni[ei == c].tolist():
            ax[col].plot([pos[c, 0], pos[v, 0]], [pos[c, 1], pos[v, 1]], color=plt.cm.tab10(c % 10), lw=3, alpha=0.18)
    ax[col].scatter(pos[:, 0], pos[:, 1], c=logpt, cmap="viridis", s=28, zorder=3)
    ax[col].set_title(f"{name}  ({n} particles)"); ax[col].set_xlabel("Delta eta"); ax[col].set_ylabel("Delta phi")
    ax[col].set_aspect("equal")
_, _, Hd_top = jet_hyperedges(POS[top_i, :int(counts[top_i])], K_HYPER)
ax[2].imshow(Hd_top, cmap="Blues", aspect="auto"); ax[2].set_title("incidence $H$ (top jet)")
ax[2].set_xlabel("hyperedge"); ax[2].set_ylabel("particle")
deg = Hd_top.sum(1)
ax[3].hist(deg, bins=range(1, deg.max() + 2), color="tab:purple", alpha=0.85)
ax[3].set_title("node degree $d_v$ (top jet)"); ax[3].set_xlabel("# hyperedges a particle is in"); ax[3].set_ylabel("count")
plt.tight_layout(); plt.show()

## 4 · A hypergraph tagger — and the loop-closing switch

We stack AllSet layers, then pool and classify. The model has a `mode` switch that makes the closing-the-loop
point *runnable*:
- `mode="hyper"` uses the k-NN hyperedges built above (a true hypergraph network);
- `mode="deepsets"` replaces the incidence with **one hyperedge per jet containing all its particles** — and the
  V→E step becomes a single global Deep-Sets pool. The very same network **becomes a PFN (Module 1)**.

In [ ]:
class HypergraphNet(nn.Module):
    def __init__(self, in_feat, n_classes, dim=64, n_layers=3):
        super().__init__()
        self.embed = mlp(in_feat, dim)
        self.layers = nn.ModuleList([AllSetLayer(dim) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, n_classes))
    def forward(self, data, mode="hyper"):
        h = self.embed(data.x); batch = data.batch; N = h.size(0)
        if mode == "hyper":
            node_idx, edge_idx = data.hyperedge_index
            num_edges = int(edge_idx.max()) + 1
        else:  # one hyperedge per jet (all its particles) -> the V->E pool IS Deep Sets / a PFN
            node_idx, edge_idx, num_edges = torch.arange(N, device=h.device), batch, int(batch.max()) + 1
        for layer in self.layers:
            h, _ = layer(h, node_idx, edge_idx, N, num_edges)
        return self.head(global_mean_pool(h, batch))

m = HypergraphNet(in_feat=7, n_classes=2).to(device)
print("forward (hyper):   ", tuple(m(next(iter(loaders["train"])).to(device), mode="hyper").shape))
print("forward (deepsets):", tuple(m(next(iter(loaders["train"])).to(device), mode="deepsets").shape))

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

@torch.no_grad()
def evaluate(model, loader, mode="hyper"):
    model.eval(); ys, ps = [], []
    for d in loader:
        d = d.to(device); ys.append(d.y.cpu()); ps.append(F.softmax(model(d, mode), -1)[:, 1].cpu())
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return {"auc": roc_auc_score(y, p), "y": y, "p": p}

def background_rejection(y, p, eff=0.5):
    fpr, tpr, _ = roc_curve(y, p)
    return 1.0 / max(np.interp(eff, tpr, fpr), 1.0 / max(int((y == 0).sum()), 1))

def train(model, loaders, mode="hyper", epochs=15, lr=1e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        model.train()
        for d in loaders["train"]:
            d = d.to(device)
            loss = F.cross_entropy(model(d, mode), d.y)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        va = evaluate(model, loaders["val"], mode)
        print(f"epoch {ep+1:2d}: val AUC {va['auc']:.4f}")
    return model

In [ ]:
print("=== Hypergraph network (k-NN hyperedges) ===")
torch.manual_seed(0)
hyper_model = HypergraphNet(7, 2, dim=64, n_layers=3).to(device)
train(hyper_model, loaders, mode="hyper", epochs=15)
res_h = evaluate(hyper_model, loaders["test"], mode="hyper")
print(f"\nHypergraph  test AUC = {res_h['auc']:.4f}   bkg rej @ eps_S=0.5 = {background_rejection(res_h['y'], res_h['p'], 0.5):.1f}")
print("references -> PFN (M1) 0.973 | ParticleNet (M2) 0.992 | ParT (M3) 0.991")

### 4.1 · Read it like an experimentalist — the ROC curve

An AUC is a summary; the object a HEP analyst actually reads is the **background-rejection curve**
$1/\varepsilon_B$ versus signal efficiency $\varepsilon_S$ (log scale — every factor matters). We plot it for the
trained hypergraph tagger and mark the $\varepsilon_S=0.5$ working point used across the course, so this model
sits on the same axes you saw for the PFN (Module 1), ParticleNet (Module 2) and ParT (Module 3).

In [ ]:
# Background rejection vs signal efficiency for the hypergraph tagger (the plot experimentalists read).
fpr, tpr, _ = roc_curve(res_h["y"], res_h["p"])
rej50 = background_rejection(res_h["y"], res_h["p"], 0.5)
plt.figure(figsize=(5, 4))
plt.plot(tpr, 1.0 / np.clip(fpr, 1e-6, 1), color="tab:red", label=f"Hypergraph / AllSet (AUC {res_h['auc']:.3f})")
plt.scatter([0.5], [rej50], color="black", zorder=5)
plt.annotate(rf"  $1/\epsilon_B$={rej50:.0f} @ $\epsilon_S$=0.5", (0.5, rej50), fontsize=9)
plt.yscale("log"); plt.xlabel(r"signal efficiency  $\epsilon_S$ (top)")
plt.ylabel(r"background rejection  $1/\epsilon_B$")
plt.grid(True, which="both", alpha=0.3); plt.legend(); plt.title("Top tagging ROC — hypergraph network")
plt.tight_layout(); plt.show()

## 5 · Closing the loop: one hyperedge = DeepSets

Now train the **same architecture** in `mode="deepsets"` — one all-encompassing hyperedge per jet, so the V→E
step is a single global pool. This is, structurally, the **PFN from Module 1**. Comparing the two runs makes the
hierarchy concrete: the hypergraph (many local groups) should edge out the single-group DeepSets, and the
DeepSets number should land near the Module-1 PFN.

In [ ]:
print("=== Same network, ONE hyperedge per jet (== DeepSets / PFN, Module 1) ===")
torch.manual_seed(0)
ds_model = HypergraphNet(7, 2, dim=64, n_layers=3).to(device)
train(ds_model, loaders, mode="deepsets", epochs=15)
res_d = evaluate(ds_model, loaders["test"], mode="deepsets")
print(f"\nHypergraph (k-NN groups) test AUC = {res_h['auc']:.4f}")
print(f"DeepSets   (1 group/jet) test AUC = {res_d['auc']:.4f}   <- the Module-1 special case, recovered")
print(f"higher-order gain from grouping   = {res_h['auc'] - res_d['auc']:+.4f}")

### 5.1 · The two runs side by side — and a structural proof that one hyperedge *is* DeepSets

Two ways to see the loop close. First the **ROC overlay**: the many-local-groups hypergraph vs the single-group
DeepSets special case, on identical data and an identical network. Second, a **structural check** — we look inside
a `mode="deepsets"` forward pass and confirm the incidence it uses is literally *one hyperedge per jet, containing
all of that jet's particles*. No approximation and no separate model: flip a switch on the hyperedge structure and
the AllSet layer becomes the Module-1 PFN.

In [ ]:
# (a) ROC overlay: hypergraph groups vs the single-hyperedge DeepSets special case.
plt.figure(figsize=(5, 4))
for res, lbl, col in [(res_h, "Hypergraph (k-NN groups)", "tab:red"),
                      (res_d, "DeepSets (1 group / jet)", "tab:blue")]:
    fpr, tpr, _ = roc_curve(res["y"], res["p"])
    plt.plot(tpr, 1.0 / np.clip(fpr, 1e-6, 1), color=col, label=f"{lbl} (AUC {res['auc']:.3f})")
plt.yscale("log"); plt.xlabel(r"signal efficiency  $\epsilon_S$"); plt.ylabel(r"background rejection  $1/\epsilon_B$")
plt.grid(True, which="both", alpha=0.3); plt.legend(); plt.title("Hypergraph vs its DeepSets special case")
plt.tight_layout(); plt.show()

# (b) structural proof: in mode="deepsets" the incidence is exactly one all-encompassing hyperedge per jet.
bb = next(iter(loaders["test"])).to(device)
N = bb.x.size(0); edge_idx = bb.batch; num_edges = int(edge_idx.max()) + 1
sizes = torch.bincount(edge_idx).cpu()                    # particles in each deepsets-mode hyperedge
per_jet = bb.ptr.diff().cpu()                             # particles per jet, from PyG batching (independent count)
print(f"deepsets mode: {bb.num_graphs} jets  ->  {num_edges} hyperedges  (one per jet)")
print(f"every hyperedge = all of one jet's particles: {torch.equal(sizes, per_jet)}")
print(f"e.g. jet 0: {int(sizes[0])} particles in a single hyperedge  ==  a global DeepSets pool (a PFN).")

## 6 · Hypergraphs for assignment (where they really shine)

Jet tagging is a mild test — the deepest use of higher-order structure in HEP is **combinatorial assignment**:
in $t\bar t$, which jets came from which top? In $H\to b\bar b$, which two? A candidate parent is precisely a
**hyperedge** over its decay products, and scoring all candidate groupings is a hypergraph problem. **SPANet**
(Fenton et al., 2021) solves exactly this with (symmetry-aware) attention over candidate assignments — an
attention-flavoured cousin of the hypergraph view: score *groups*, not pairs. The same machinery powers
event-level reconstruction and pile-up mitigation, where the natural object is a **set of sets**.

### 6.1 · A runnable toy: assignment = scoring *groups* (hyperedges)

Let us make "score groups, not pairs" concrete. We simulate a toy $t\bar t\to 6$ jets event: two tops, each
decaying to **three** jets whose invariant mass is $m_t\approx 173$ GeV *by construction*; we boost each top,
smear the jets, and shuffle all six. The task is the classic **assignment**: recover which three jets came from
each top. There are only $\binom{6}{3}/2 = 10$ candidate 3+3 partitions, and **each candidate parent is a
hyperedge** over its three decay products. The right score of a group is a **permutation-invariant function of its
members** — exactly a V→E Deep Set. We compare three scorers:

1. **random** — pick a partition at random ($1/10$);
2. **analytic** — score a triplet by how close its invariant mass is to $m_t$ (the physics answer);
3. **learned V→E Deep Set** — train $\rho\!\big(\sum_i \phi(j_i)\big)$ on triplets and pick the partition it ranks highest.

Watch the gap between (2) and (3): the analytic score *bakes in the Minkowski-mass invariant*, while the network
must discover it from data — the very Module-4 lesson, resurfacing at the capstone.

In [ ]:
# Toy tt-bar: two tops -> 3 jets each (invariant mass = m_t by construction), boosted, smeared, shuffled.
from itertools import combinations
M_TOP = 173.0
def make_ttbar(n_events, seed=0, smear=3.0):
    rng = np.random.default_rng(seed); Ev, Truth = [], []
    for _ in range(n_events):
        jets, who = [], []
        for t in range(2):
            p = rng.normal(size=(3, 3)); p -= p.mean(0)             # 3 momenta with zero total 3-momentum
            p *= M_TOP / np.linalg.norm(p, axis=1).sum()            # => invariant mass of the sum = m_t exactly
            E = np.linalg.norm(p, axis=1, keepdims=True)            # massless jets: E = |p|
            four = np.concatenate([E, p], 1)
            d = rng.normal(size=3); d /= np.linalg.norm(d); b = rng.uniform(0.2, 0.75)   # |beta| < 0.8 -> finite gamma
            beta = b * d; g = 1 / np.sqrt(1 - b * b); bb = (beta * beta).sum()
            L = np.eye(4); L[0, 0] = g; L[0, 1:] = g * beta; L[1:, 0] = g * beta
            L[1:, 1:] = np.eye(3) + (g - 1) * np.outer(beta, beta) / bb
            four = four @ L.T + rng.normal(0, smear, (3, 4))        # boost the top, then detector smearing
            jets += list(four); who += [t, t, t]
        idx = rng.permutation(6)
        Ev.append(np.array(jets)[idx].astype(np.float32)); Truth.append(np.array(who)[idx])
    return Ev, Truth

def mass(four_sum):                                                 # invariant mass of a summed 4-vector
    return np.sqrt(np.clip(four_sum[..., 0] ** 2 - (four_sum[..., 1:] ** 2).sum(-1), 0, None))

PARTS = [(frozenset(c), frozenset(range(6)) - frozenset(c)) for c in combinations(range(6), 3) if 0 in c]  # 10 splits
def truth_group(who): return frozenset(np.where(who == who[0])[0].tolist())

Ev_te, Tr_te = make_ttbar(600, seed=1)
# the signal: correct triplets peak at m_t, wrong (mixed) triplets do not
cm = [mass(j[list(truth_group(w))].sum(0)) for j, w in zip(Ev_te, Tr_te)]
wm = [mass(j[list(next(P[0] for P in PARTS if P[0] != truth_group(w) and P[1] != truth_group(w)))].sum(0))
      for j, w in zip(Ev_te, Tr_te)]
plt.figure(figsize=(5, 3.4))
plt.hist(cm, bins=40, range=(80, 320), alpha=0.7, label="correct triplet")
plt.hist(wm, bins=40, range=(80, 320), alpha=0.7, label="wrong (mixed) triplet")
plt.axvline(M_TOP, color="k", ls="--", lw=1, label=r"$m_t$ = 173")
plt.xlabel("triplet invariant mass [GeV]"); plt.ylabel("events"); plt.legend()
plt.title("why grouping is learnable"); plt.tight_layout(); plt.show()

def analytic_accuracy(Ev, Tr):
    ok = 0
    for j, w in zip(Ev, Tr):
        tg = truth_group(w); best, bg = 1e18, None
        for g0, g1 in PARTS:
            s = abs(mass(j[list(g0)].sum(0)) - M_TOP) + abs(mass(j[list(g1)].sum(0)) - M_TOP)
            if s < best: best, bg = s, g0
        ok += (bg == tg)
    return ok / len(Ev)
print(f"random baseline (1 of 10)          : {1 / len(PARTS):.3f}")
print(f"analytic (Minkowski-mass) accuracy : {analytic_accuracy(Ev_te, Tr_te):.3f}")

In [ ]:
# Learned V->E Deep Set scorer: rho( sum_i phi(jet_i) ) over a candidate triplet, trained to rank the true partition #1.
class GroupScorer(nn.Module):
    def __init__(self, dim=64):
        super().__init__(); self.embed = mlp(4, dim); self.phi = mlp(dim, dim); self.rho = mlp(dim, 1)
    def encode(self, j): return self.embed(j * 0.02)                        # scale O(100 GeV) 4-vectors to O(1)
    def group(self, h, idxs): return self.rho(self.phi(h[:, idxs]).sum(1)).squeeze(-1)     # V->E DeepSet of a group
    def partition_logits(self, h):                                          # (B,10): total score of each 3+3 split
        return torch.stack([self.group(h, sorted(g0)) + self.group(h, sorted(g1)) for g0, g1 in PARTS], 1)

Ev_tr, Tr_tr = make_ttbar(1500, seed=0)
Xtr = torch.tensor(np.stack(Ev_tr)).to(device); Xte = torch.tensor(np.stack(Ev_te)).to(device)
lab = lambda w: [k for k, P in enumerate(PARTS) if P[0] == truth_group(w)][0]              # index of the true partition
ytr = torch.tensor([lab(w) for w in Tr_tr]).to(device); yte = torch.tensor([lab(w) for w in Tr_te])

torch.manual_seed(0); scorer = GroupScorer().to(device)
opt = torch.optim.AdamW(scorer.parameters(), 3e-3, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=25)
for ep in range(25):
    perm = torch.randperm(len(Ev_tr))
    for s in range(0, len(Ev_tr), 128):
        b = perm[s:s + 128]
        loss = F.cross_entropy(scorer.partition_logits(scorer.encode(Xtr[b])), ytr[b])     # listwise: rank truth #1
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(scorer.parameters(), 5.0); opt.step()
    sch.step()
with torch.no_grad():
    pred = scorer.partition_logits(scorer.encode(Xte)).argmax(1).cpu()
learned = (pred == yte).float().mean().item()
print(f"random baseline (1 of 10)          : {1 / len(PARTS):.3f}")
print(f"learned V->E DeepSet scorer        : {learned:.3f}   ({learned * len(PARTS):.1f}x random)")
print("The learned group-scorer beats random handily; the analytic mass score does better still --")
print("bake in the invariant (Module 4). Exercise 5 adds a SPANet-style attention baseline and scales this up.")

## 7 · From hyperedges to ranked cells: the combinatorial complex (TopoNetX)

A hypergraph treats every group the same — it is *flat*. But jet structure is **hierarchical**: particles cluster
into **subjets** (the prongs of the decay), and subjets make up the **jet**. We want one object that holds groups
*at several scales at once*, together with a notion of which group sits inside which. That object is the
**combinatorial complex (CC)** — the central data structure of *topological deep learning* (Hajij et al., 2023),
provided by the **TopoNetX** library.

**Definition.** A combinatorial complex is a triple $(S,\mathcal X,\mathrm{rk})$: a set of nodes $S$, a collection
$\mathcal X$ of non-empty **cells** (subsets of $S$), and a **rank** function $\mathrm{rk}:\mathcal X\to\mathbb Z_{\ge 0}$ with

1. every singleton is a rank-0 cell: $\{s\}\in\mathcal X$ and $\mathrm{rk}(\{s\})=0$;
2. rank respects containment: $x\subseteq y \Rightarrow \mathrm{rk}(x)\le \mathrm{rk}(y)$.

So each cell carries a **rank** (a scale), and larger-scale cells have higher rank. That single idea unifies the
whole course:

| object | cells | ranks | closed under subsets? |
|---|---|---|---|
| **graph** | nodes + **size-2** edges | 0, 1 | — |
| **hypergraph** | nodes + arbitrary-size hyperedges | 0, **1 only** | no |
| **simplicial / cell complex** | ranked cells | 0,1,2,… | **yes** (must contain all faces) |
| **combinatorial complex** | **arbitrary-size** cells, **any rank** | 0,1,2,… | **no** |

A **CC keeps the best of both**: arbitrary-size groups (like a hypergraph) *and* a rank hierarchy (like a cell
complex), with **no** closure requirement. A hypergraph is exactly the special case in which every non-node cell
has rank 1 — so the CC **generalizes** §3–§4. Our jet CC has three ranks,
$$\underbrace{\text{particles}}_{\text{rank }0}\ \subset\ \underbrace{\text{subjets}}_{\text{rank }1}\ \subset\ \underbrace{\text{the jet}}_{\text{rank }2},$$
the "set of sets" of §6 promoted to a genuine topological object. TopoNetX will build it, hand us its
incidence/adjacency matrices, and let us message-pass across the ranks.

In [ ]:
# Build ONE jet's combinatorial complex with TopoNetX: particles (r0) -> subjets (r1) -> jet (r2).
from toponetx.classes import CombinatorialComplex

def subjet_partition(pos, n_sub=3):
    """Balanced k-means on (eta,phi) -> >=2 subjets, each with >=2 particles (so every rank-1 cell is valid)."""
    n = len(pos); k = max(2, min(n_sub, n // 2))                    # feasible: >=2 members each when n>=4
    C = pos[:k].astype(float).copy()
    for _ in range(8):                                             # Lloyd iterations from leading-k seeds
        A = np.argmin(((pos[:, None] - C[None]) ** 2).sum(-1), 1)
        for c in range(k):
            if (A == c).any(): C[c] = pos[A == c].mean(0)
    A = np.argmin(((pos[:, None] - C[None]) ** 2).sum(-1), 1)
    for _ in range(6 * k):                                         # repair: force every subjet to have >=2 members
        sizes = np.bincount(A, minlength=k); small = np.where(sizes < 2)[0]
        if len(small) == 0: break
        c = small[0]; big = sizes.argmax(); mem = np.where(A == big)[0]
        A[mem[np.argmin(((pos[mem] - C[c]) ** 2).sum(-1))]] = c; C[c] = pos[A == c].mean(0)
    relabel = {v: i for i, v in enumerate(sorted(set(A.tolist())))}
    return np.array([relabel[a] for a in A], dtype=int), len(relabel)

def make_cc(pos, n_sub=3):
    """Assemble a TopoNetX CombinatorialComplex for a jet from its particle (eta,phi) positions."""
    n = len(pos); assign, ns = subjet_partition(pos, n_sub)
    cc = CombinatorialComplex()
    for p in range(n):  cc.add_cell([p], rank=0)                                                   # rank 0: particles
    for s in range(ns): cc.add_cell([int(p) for p in range(n) if assign[p] == s], rank=1)          # rank 1: subjets
    cc.add_cell(list(range(n)), rank=2)                                                            # rank 2: the jet
    return cc, assign, ns

ex = next(i for i in range(len(labels)) if labels[i] == 1 and counts[i] >= 25)     # an example top jet
ex_n = int(counts[ex]); ex_pos = POS[ex, :ex_n]
cc, ex_assign, ex_ns = make_cc(ex_pos, n_sub=3)
print(cc)
print("cells per rank (shape):", cc.shape, "   ranks present:", cc.ranks)
print(f"rank 0: {ex_n} particles    rank 1: {ex_ns} subjets    rank 2: 1 jet")
print("first subjet (a rank-1 cell) has particles:", sorted(int(p) for p in list(cc.skeleton(1))[0]))

### 7.1 · Incidence, adjacency, and message passing across ranks

The structure of a CC is captured by its **incidence matrices** between consecutive ranks — the direct
generalization of the hypergraph incidence $H$ of §1:
$$B_{0,1}\in\{0,1\}^{\,|\text{particles}|\times|\text{subjets}|},\qquad B_{1,2}\in\{0,1\}^{\,|\text{subjets}|\times 1},$$
with $(B_{0,1})_{ve}=1$ iff particle $v$ lies in subjet $e$, and $(B_{1,2})_{ec}=1$ iff subjet $e$ lies in the jet
$c$. From them one forms **adjacency** matrices, e.g. $A_0 = B_{0,1}B_{0,1}^{\top}$ (off-diagonal): two particles
are *adjacent* iff they share a subjet. TopoNetX computes all of these with `.incidence_matrix(r, r')` and
`.adjacency_matrix(r, via_rank=r')`.

**Higher-order message passing (a CCNN; Hajij et al., 2023)** flows information *up and down the rank ladder*, and
— crucially — **every step is a Deep Set over faces**, the Module-1 atom again:
$$
\textbf{up:}\quad z_e=\rho\Big(\!\sum_{v\in e}\phi(h_v)\Big),\ \ w=\rho\Big(\!\sum_{e\subset c}\phi(z_e)\Big);
\qquad
\textbf{down:}\quad h_v \mathrel{+}= \rho\Big(\!\sum_{e\ni v}\phi(z_e)\Big),\ \ z_e \mathrel{+}= \rho\Big(\!\sum_{c\supset e}\phi(w)\Big).
$$
In words: a subjet summarizes its particles and the jet summarizes its subjets (**up**, 0→1→2); then the jet
informs its subjets and each subjet informs its particles (**down**, 2→1→0). A CC layer is therefore **AllSet
lifted to a hierarchy** — the two-nested-DeepSets of §2, now stacked across ranks. We build it by reusing the very
same `DeepSetsAgg` atom from §2.

In [ ]:
# Visualize the example jet's combinatorial complex (every matrix comes from TopoNetX).
from scipy.spatial import ConvexHull
B01 = np.asarray(cc.incidence_matrix(0, 1).todense())            # (particles x subjets)
A0  = np.asarray(cc.adjacency_matrix(0, via_rank=1).todense())   # (particles x particles): do they share a subjet?
o   = np.argsort(ex_assign)                                      # order particles by subjet, so blocks are visible
cmap = plt.cm.tab10
fig, ax = plt.subplots(1, 4, figsize=(16, 4))

# (A) particles in eta-phi, coloured by subjet, a hull per subjet, and the jet as a bounding box
for s in range(ex_ns):
    m = np.where(ex_assign == s)[0]; pts = ex_pos[m]
    ax[0].scatter(pts[:, 0], pts[:, 1], s=32, color=cmap(s), label=f"subjet {s}", zorder=3)
    if len(m) >= 3:
        hull = ConvexHull(pts); loop = np.append(hull.vertices, hull.vertices[0])
        ax[0].fill(pts[loop, 0], pts[loop, 1], color=cmap(s), alpha=0.15)
lo, hi = ex_pos.min(0) - 0.03, ex_pos.max(0) + 0.03
ax[0].add_patch(plt.Rectangle(lo, *(hi - lo), fill=False, ls="--", ec="k", lw=1.2))
ax[0].set_title("rank 0 -> 1 -> 2\nparticles / subjets / jet"); ax[0].set_xlabel("Delta eta"); ax[0].set_ylabel("Delta phi")
ax[0].legend(fontsize=7); ax[0].set_aspect("equal")

# (B) the rank ladder (a Hasse diagram) with incidence edges
xp = np.linspace(0, 1, ex_n); xs = np.linspace(0.25, 0.75, ex_ns)
for v in range(ex_n):
    ax[1].plot([xp[v], xs[ex_assign[v]]], [0, 1], color=cmap(ex_assign[v]), lw=0.6, alpha=0.5)
for s in range(ex_ns):
    ax[1].plot([xs[s], 0.5], [1, 2], color="gray", lw=1.3)
ax[1].scatter(xp, np.zeros(ex_n), s=16, c=[cmap(a) for a in ex_assign], zorder=3)
ax[1].scatter(xs, np.ones(ex_ns), s=110, marker="s", c=[cmap(s) for s in range(ex_ns)], zorder=3)
ax[1].scatter([0.5], [2], s=180, marker="^", c="k", zorder=3)
ax[1].set_yticks([0, 1, 2]); ax[1].set_yticklabels(["rank 0\nparticles", "rank 1\nsubjets", "rank 2\njet"])
ax[1].set_xticks([]); ax[1].set_title("the combinatorial complex\n(incidence / Hasse diagram)")

# (C) incidence B_{0,1} and (D) adjacency A_0, both from TopoNetX, particles ordered by subjet
ax[2].imshow(B01[o], cmap="Blues", aspect="auto"); ax[2].set_title(r"incidence $B_{0,1}$"); ax[2].set_xlabel("subjet"); ax[2].set_ylabel("particle")
ax[3].imshow(A0[o][:, o], cmap="Purples", aspect="auto"); ax[3].set_title(r"adjacency $A_0$ (share a subjet)"); ax[3].set_xlabel("particle"); ax[3].set_ylabel("particle")
plt.tight_layout(); plt.show()

In [ ]:
# Build the CC dataset with TopoNetX, bridging each complex to torch index tensors for batched message passing.
import time
class CCData(Data):
    """A combinatorial-complex sample. inc01 = B_{0,1} and inc12 = B_{1,2} as index lists; __inc__ offsets on batching."""
    def __cat_dim__(self, key, value, *args, **kwargs):
        return -1 if key in ("inc01", "inc12") else super().__cat_dim__(key, value, *args, **kwargs)
    def __inc__(self, key, value, *args, **kwargs):
        if key == "inc01": return torch.tensor([[self.num_nodes], [self.num_subjets]])
        if key == "inc12": return torch.tensor([[self.num_subjets], [self.num_jets]])
        return super().__inc__(key, value, *args, **kwargs)

def build_cc_data(x, pos, label, n_sub=3):
    cc, assign, ns = make_cc(pos, n_sub); n = x.shape[0]
    r0, r1, B01 = cc.incidence_matrix(0, 1, index=True)                     # TopoNetX incidence + index dicts
    B01 = np.asarray(B01.todense()); row_to_part = {i: next(iter(fs)) for fs, i in r0.items()}
    rows, cols = np.nonzero(B01)                                            # (particle-row, subjet-col) memberships
    p_idx = np.array([row_to_part[r] for r in rows])                       # align incidence rows to particle order
    inc01 = torch.tensor(np.stack([p_idx, cols]), dtype=torch.long)        # B_{0,1} as an index list
    inc12 = torch.tensor(np.stack([np.arange(ns), np.zeros(ns, int)]), dtype=torch.long)   # B_{1,2}: every subjet in the one jet
    d = CCData(x=torch.from_numpy(x), inc01=inc01, inc12=inc12, y=torch.tensor([label]))
    d.num_subjets = ns; d.num_jets = 1
    return d

t0 = time.time()
cc_list = [build_cc_data(X[i, :int(counts[i])], POS[i, :int(counts[i])], int(labels[i]))
           for i in range(len(labels)) if counts[i] >= 4]
print(f"built {len(cc_list)} combinatorial complexes with TopoNetX in {time.time() - t0:.1f}s")
random.seed(0); random.shuffle(cc_list)
ncc = len(cc_list); acc, bcc = int(0.70 * ncc), int(0.85 * ncc)
cc_loaders = {"train": DataLoader(cc_list[:acc], batch_size=64, shuffle=True),
              "val":   DataLoader(cc_list[acc:bcc], batch_size=128),
              "test":  DataLoader(cc_list[bcc:], batch_size=128)}
_b = next(iter(cc_loaders["train"]))
print(f"batch: {_b.num_graphs} jets, {_b.x.shape[0]} particles, {int(_b.inc01[1].max())+1} subjets, {int(_b.inc12[1].max())+1} jets")

In [ ]:
# A combinatorial-complex network: message passing up (0->1->2) and down (2->1->0), reusing the §2 DeepSets atom.
class CCLayer(nn.Module):
    """One CC layer = DeepSets pools across the rank ladder (particles <-> subjets <-> jet)."""
    def __init__(self, dim):
        super().__init__()
        self.up01 = DeepSetsAgg(dim, dim); self.up12 = DeepSetsAgg(dim, dim)      # up:   V->E, E->C
        self.dn21 = mlp(dim, dim); self.dn10 = mlp(dim, dim)                       # down: C->E, E->V
        self.n0 = nn.LayerNorm(dim); self.n1 = nn.LayerNorm(dim)
    def forward(self, h0, pidx, sidx, s2idx, jidx, Ns, Nj):
        h1 = self.up01(h0, pidx, sidx, Ns)                     # each subjet pools its particles   (0 -> 1)
        h2 = self.up12(h1, s2idx, jidx, Nj)                    # the jet pools its subjets          (1 -> 2)
        jet_of_sub = torch.zeros(Ns, dtype=torch.long, device=h0.device); jet_of_sub[s2idx] = jidx
        h1 = self.n1(h1 + self.dn21(h2)[jet_of_sub])           # jet informs its subjets            (2 -> 1)
        h0 = self.n0(h0 + self.dn10(h1)[sidx])                 # each subjet informs its particles  (1 -> 0)
        return h0, h2, jet_of_sub

class CCNet(nn.Module):
    def __init__(self, in_feat, n_classes=2, dim=64, n_layers=3):
        super().__init__()
        self.embed = mlp(in_feat, dim)
        self.layers = nn.ModuleList([CCLayer(dim) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(2 * dim), nn.Linear(2 * dim, dim), nn.SiLU(), nn.Linear(dim, n_classes))
    def forward(self, data):
        h0 = self.embed(data.x); pidx, sidx = data.inc01; s2idx, jidx = data.inc12
        Ns = int(sidx.max()) + 1; Nj = int(jidx.max()) + 1
        for layer in self.layers:
            h0, h2, jet_of_sub = layer(h0, pidx, sidx, s2idx, jidx, Ns, Nj)
        return self.head(torch.cat([h2, global_mean_pool(h0, jet_of_sub[sidx])], -1))   # jet-cell + pooled particles

@torch.no_grad()
def evaluate_cc(model, loader):
    model.eval(); ys, ps = [], []
    for d in loader:
        d = d.to(device); ys.append(d.y.cpu()); ps.append(F.softmax(model(d), -1)[:, 1].cpu())
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return {"auc": roc_auc_score(y, p), "y": y, "p": p}

print("CCNet forward:", tuple(CCNet(X.shape[-1]).to(device)(next(iter(cc_loaders["train"])).to(device)).shape))

In [ ]:
print("=== Combinatorial-complex network (particles -> subjets -> jet) ===")
torch.manual_seed(0)
cc_model = CCNet(X.shape[-1], dim=64, n_layers=3).to(device)
opt = torch.optim.AdamW(cc_model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=12)
for ep in range(12):
    cc_model.train()
    for d in cc_loaders["train"]:
        d = d.to(device); loss = F.cross_entropy(cc_model(d), d.y)
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(cc_model.parameters(), 5.0); opt.step()
    sched.step()
    if (ep + 1) % 3 == 0: print(f"epoch {ep+1:2d}: val AUC {evaluate_cc(cc_model, cc_loaders['val'])['auc']:.4f}")
res_cc = evaluate_cc(cc_model, cc_loaders["test"])
print(f"\nCombinatorial complex  test AUC = {res_cc['auc']:.4f}   bkg rej @ eps_S=0.5 = {background_rejection(res_cc['y'], res_cc['p'], 0.5):.1f}")
print(f"compare ->  Hypergraph (§4) {res_h['auc']:.4f}  |  DeepSets special case (§5) {res_d['auc']:.4f}")

In [ ]:
# ROC: the combinatorial-complex tagger vs the hypergraph and DeepSets models (same task, same axes).
plt.figure(figsize=(5.4, 4.2))
for res, lbl, col in [(res_h, "Hypergraph / AllSet (§4)", "tab:red"),
                      (res_cc, "Combinatorial complex (§7)", "tab:green"),
                      (res_d, "DeepSets special case (§5)", "tab:blue")]:
    fpr, tpr, _ = roc_curve(res["y"], res["p"])
    plt.plot(tpr, 1.0 / np.clip(fpr, 1e-6, 1), color=col, label=f"{lbl}  (AUC {res['auc']:.3f})")
plt.yscale("log"); plt.xlabel(r"signal efficiency  $\epsilon_S$ (top)"); plt.ylabel(r"background rejection  $1/\epsilon_B$")
plt.grid(True, which="both", alpha=0.3); plt.legend(); plt.title("Top tagging: hypergraph vs combinatorial complex")
plt.tight_layout(); plt.show()

### 7.2 · What the ranks buy — and the generalization ladder

The combinatorial-complex tagger reaches an AUC close to the hypergraph's while carrying an explicit,
**interpretable hierarchy**: a subjet-level representation (rank 1) and a jet-level one (rank 2) that a physicist
can read off directly. Where a hypergraph gives one flat pool of overlapping groups, the CC gives a **ladder of
scales** with messages flowing both ways — particles inform subjets inform the jet, and the global jet context
flows back down to refine each particle. And none of the machinery is new: **every up/down step is the Module-1
DeepSets pool**, so a CC layer is literally *AllSet applied at each rank interface*.

This completes the generalization ladder the course has climbed:

| relations | object | where |
|---|---|---|
| none | a **set** | Module 1 (DeepSets) |
| pairs | a **graph** | Module 2 |
| soft pairs | attention (a complete graph) | Module 3 |
| groups | a **hypergraph** | §1–§6 |
| **ranked groups** | a **combinatorial complex** | **§7 (here)** |

Topological deep learning (Hajij et al., *Topological Deep Learning: Going Beyond Graph Data*, 2023; the
**TopoX**/TopoNetX software suite, 2024) studies exactly this most general object: simplicial complexes, cell
complexes, hypergraphs and combinatorial complexes are all special cases, and message passing on every one of them
is — once more — the permutation-invariant Deep Sets aggregation you built in Module 1. *One atom, the entire
ladder.*

**Refs:** Hajij et al. *Topological Deep Learning* 2206.00606; the **TopoX** suite 2402.02441; Feng *HGNN*
1809.09401; Chien *AllSet* 2106.13264.

## 8 · Capstone: one idea, five modules

Everything in this course was **constrained message passing** — a choice of *relations*, an *aggregation*, and a
*symmetry*. The whole arc on one page:

| Module | Relations | Aggregation | Symmetry baked in | Flagship |
|---|---|---|---|---|
| **1 · DeepSets** | none (independent particles) | one global pool | permutation | PFN / EFN |
| **2 · GNN** | k-NN **pairs** | over neighbours | permutation | ParticleNet |
| **3 · Attention** | **soft** all-to-all pairs | attention-weighted | permutation (+ physics bias) | ParT |
| **4 · Equivariant** | pairs of 4-vectors | over **invariants** | permutation **+ Lorentz** | LorentzNet / PELICAN |
| **5 · Hypergraph** | **groups** (hyperedges) | two nested DeepSets | permutation | AllSet / HGNN |

Read top to bottom: we added one ingredient at a time — **edges**, then **soft** edges, then a **symmetry**
constraint, then **higher-order** edges. And the **aggregation never changed**: it is always the Deep Sets pool
$\rho(\sum\phi(\cdot))$ from Module 1. A GNN is a DeepSet per neighbourhood; attention is a DeepSet with soft
weights; an equivariant net is a DeepSet over invariants; a hypergraph layer is **two** DeepSets. *One atom, five
models.*

### 8.1 · The whole course as one figure

The table above, drawn. We plot the top-tagging AUC of the two models we trained *in this notebook* — the
hypergraph tagger and its DeepSets special case — next to the reference numbers for the earlier flagships
(ParticleNet, ParT). One aggregation — $\rho(\sum\phi(\cdot))$ — underlies every bar; what changes along the axis
is only the **relations** each model is allowed (none → pairs → soft pairs → groups).

In [ ]:
# One figure for the whole course: AUC vs the relational structure each model uses.
bars = [("DeepSets / PFN\n(no relations)", res_d["auc"],  "tab:blue"),
        ("ParticleNet\n(k-NN pairs)",       0.992,         "tab:green"),
        ("ParT\n(soft pairs)",              0.991,         "tab:olive"),
        ("Hypergraph / AllSet\n(groups)",   res_h["auc"],  "tab:red"),
        ("Comb. complex\n(ranked groups)",  res_cc["auc"], "tab:purple")]
names = [b[0] for b in bars]; vals = [b[1] for b in bars]; cols = [b[2] for b in bars]
plt.figure(figsize=(9, 3.9))
plt.bar(names, vals, color=cols, alpha=0.85)
for i, v in enumerate(vals):
    plt.text(i, v + 0.0004, f"{v:.3f}", ha="center", fontsize=9)
plt.ylim(0.96, 1.0); plt.ylabel("top-tagging AUC")
plt.title("One aggregation, many relations — the course on one axis")
plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()
print("The same DeepSets atom sits under every bar; only the *relations* (and the symmetry) change. The loop is closed.")

## 9 · Outlook: foundation models for particle physics

Where the field is going (2024–): instead of training a fresh tagger per task, **pre-train one large
set/graph/transformer backbone** on huge unlabelled jet/event samples with a self-supervised objective, then
fine-tune. Lines to watch:
- **Masked particle modelling** — hide some particles, predict them (BERT for jets).
- **OmniJet-style** generative/autoregressive jet models and cross-task **backbones** (tag, generate, reconstruct
  from one model).
- **Contrastive / symmetry-aware pre-training** that bakes in the very inductive biases from Modules 3–4.

The building blocks are exactly what you implemented: permutation-invariant aggregation, attention, equivariance,
and higher-order structure. You now have the vocabulary to read — and build — the next generation.

## 10 · Exercises

1. **Subjet hyperedges.** Replace k-NN hyperedges with **re-clustered subjets** (e.g. assign each particle to its
   nearest of the 3 leading-$p_T$ particles). Do physically-motivated groups beat k-NN groups for top tagging?
2. **Hyperedge size.** Scan `K_HYPER` (2, 4, 8, 16). At $k\!=\!1$ you approach a graph; at $k\!=\!N$ a single
   DeepSet. Where is the sweet spot, and why?
3. **Recover a GNN.** Make every hyperedge a **pair** $\{i,j\}$ and confirm AllSet reproduces Module-2 message
   passing. (You've now shown DeepSets, GNNs *and* hypergraphs are one family.)
4. **Sets of sets.** Build an **event** hypergraph: nodes = particles, hyperedges = jets, plus one event-level
   hyperedge. Classify events and compare to flattening everything into one jet.
5. **Assignment.** Set up a toy $t\bar t$ jet→parent assignment and score candidate groupings with the V→E head;
   compare to a SPANet-style attention baseline.
6. **Attention pooling inside AllSet.** Swap the sum-pool in `DeepSetsAgg` for the attention pooling from Module 3
   (this is literally the *AllSet-Transformer* variant).

## 11 · The end — and the beginning

You built, from scratch and trained on real LHC jets, the full modern toolkit for learning on particle data:
**DeepSets → GNNs → Transformers → equivariant networks → hypergraph networks**, all as one idea — *constrained
message passing on sets*. You verified the symmetries numerically, evaluated like an experimentalist (AUC,
background rejection), and saw each model as a small variation on the Deep Sets atom.

The same toolkit now reaches far beyond tagging: calorimeter reconstruction, tracking, pile-up mitigation,
anomaly detection, simulation and generation, and the foundation models that will tie them together. Go build.